# DeepBio-Scan: Large-Scale Atlas Seeding (N=500,000)
## Phase 1: Reference Atlas Generation (Sharded 5x100k)

**Personas Active:**
- `@Embedder-ML` (Model Logic & Inference)
- `@Data-Ops` (Data Pipeline & Parquet Export)

**Hardware Target:** Google Colab T4/A100 GPU

# DeepBio-Scan: Large-Scale Atlas Seeding (N=500,000)
## Phase 1: Marine Atlas Generation for 2TB Roadmap

**Output Contract:**
- `shard_1.parquet` ... `shard_5.parquet` (100,000 records each)
- L2-normalized 768-d vectors
- `metagenomic_source='NCBI-Nucleotide'`

In [ ]:
# @Data-Ops: Dependency Setup
!pip uninstall -y torch_xla
!pip install --upgrade transformers==4.40.2 pandas pyarrow duckdb lancedb accelerate biopython tqdm

In [ ]:
import os
import time
import torch
import pandas as pd
import numpy as np
from Bio import Entrez, SeqIO
from transformers import AutoTokenizer, AutoModelForMaskedLM, AutoConfig
from tqdm.auto import tqdm

# @Embedder-ML: GPU Acceleration Check
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Running on: {device.upper()}")
if device == "cpu":
    print("WARNING: GPU not detected. Embedding will be slow.")

In [ ]:
# @Data-Ops: Step 1 - High-Speed Batch Fetching (500k, 5 shards x 100k, 7-rank taxonomy)
# TITANIUM V2: Batch 50, Memory Guards, Pre-Fetch Cleanup
import io
import time
import shutil
import gc
import pyarrow as pa
import pyarrow.parquet as pq

# 1. CRITICAL MEMORY CLEANUP BEFORE START
if 'embedder' in globals():
    print("⚠️ Releasing 'embedder' model usage to free RAM for data retrieval...")
    try:
        del embedder
    except:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

Entrez.email = "data-ops@deepbio.scan"
TARGET_COUNT = 500_000
SHARD_SIZE = 100_000
FETCH_BATCH_SIZE = 50   # Atomic batch size
SHARDS_DIR = "shards"
FORCE_REBUILD = True

if FORCE_REBUILD and os.path.exists(SHARDS_DIR):
    print(f"FORCE_REBUILD=True: Cleaning up {SHARDS_DIR}...")
    try:
        shutil.rmtree(SHARDS_DIR)
    except OSError as e:
        print(f"Warning cleaning dir: {e}")

os.makedirs(SHARDS_DIR, exist_ok=True)

RANKS = ["Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"]
MAX_RETRIES = 5
RETRY_BASE_DELAY = 1

def _rank_aware_fill(rank_values):
    out = {}
    for rank in RANKS:
        val = str(rank_values.get(rank, "")).strip()
        if not val or val.lower() in {"unknown", "nan", "none"}:
            out[rank] = f"[Unclassified {rank}]"
        else:
            out[rank] = val
    return out

def fetch_taxonomy_metadata(tax_ids, retry_count=0):
    if not tax_ids: return {}
    tax_dict = {}
    try:
        handle = Entrez.efetch(db="taxonomy", id=",".join(tax_ids), retmode="xml")
        records = Entrez.read(handle)
        handle.close()
        for record in records:
            tax_id = str(record.get("TaxId", "Unknown"))
            rank_values = {r: "" for r in RANKS}
            rec_rank = str(record.get("Rank", "")).strip().lower()
            rec_name = str(record.get("ScientificName", "")).strip()
            if rec_rank in {r.lower() for r in RANKS} and rec_name:
                rank_values[rec_rank.capitalize()] = rec_name
            for taxon in record.get("LineageEx", []):
                rank = str(taxon.get("Rank", "")).strip().lower()
                name = str(taxon.get("ScientificName", "")).strip()
                if rank in {r.lower() for r in RANKS} and name:
                    rank_values[rank.capitalize()] = name
            tax_dict[tax_id] = _rank_aware_fill(rank_values)
        del records
        return tax_dict
    except Exception as e:
        if retry_count < MAX_RETRIES:
            time.sleep(RETRY_BASE_DELAY * (2 ** retry_count))
            return fetch_taxonomy_metadata(tax_ids, retry_count + 1)
        print(f"Taxonomy fetch failed: {e}")
    return {}

def fetch_batch_sequences(start, retmax, webenv, query_key, retry_count=0):
    try:
        # Use simple text mode fetch
        fasta_handle = Entrez.efetch(
            db="nucleotide", 
            rettype="fasta", 
            retmode="text", 
            retstart=start, 
            retmax=retmax, 
            webenv=webenv, 
            query_key=query_key
        )
        # Check size before reading all if possible? No simple way with handle.
        # Just read. For 50 items it should be small.
        blob = fasta_handle.read()
        fasta_handle.close()
        
        if isinstance(blob, bytes): 
            blob = blob.decode('utf-8', errors='replace')
            
        res = list(SeqIO.parse(io.StringIO(blob), "fasta"))
        del blob
        return res
    except Exception as e:
        if retry_count < MAX_RETRIES:
            time.sleep(RETRY_BASE_DELAY * (2 ** retry_count))
            return fetch_batch_sequences(start, retmax, webenv, query_key, retry_count + 1)
        print(f"Seq fetch failed: {e}")
        return None

def fetch_batch_metadata(start, retmax, webenv, query_key, retry_count=0):
    try:
        sh = Entrez.esummary(db="nucleotide", retstart=start, retmax=retmax, webenv=webenv, query_key=query_key)
        res = Entrez.read(sh)
        sh.close()
        return res
    except Exception as e:
        if retry_count < MAX_RETRIES:
            time.sleep(RETRY_BASE_DELAY * (2**retry_count))
            return fetch_batch_metadata(start, retmax, webenv, query_key, retry_count + 1)
        return None

def fetch_marine_eukaryotes_500k():
    print("Initiating Entrez Search...")
    query = "eukaryota[Organism] AND (marine[All Fields] OR ocean[All Fields] OR benthic[All Fields]) AND (18S[All Fields] OR COI[All Fields])"
    handle = Entrez.esearch(db="nucleotide", term=query, retmax=TARGET_COUNT, usehistory="y")
    record = Entrez.read(handle)
    handle.close()
    
    total_found = int(record["Count"])
    webenv = record["WebEnv"]
    query_key = record["QueryKey"]
    print(f"Found {total_found:,} records.")

    schema = pa.schema([
        ('AccessionID', pa.string()), ('ScientificName', pa.string()), ('TaxID', pa.string()),
        ('Kingdom', pa.string()), ('Phylum', pa.string()), ('Class', pa.string()),
        ('Order', pa.string()), ('Family', pa.string()), ('Genus', pa.string()),
        ('Species', pa.string()), ('Sequence', pa.string()), 
        ('metagenomic_source', pa.string()), ('Quality_Check', pa.bool_())
    ])

    shard_paths = []
    
    for shard_idx in range(5):
        shard_start = shard_idx * SHARD_SIZE
        shard_end = min(shard_start + SHARD_SIZE, min(TARGET_COUNT, total_found))
        if shard_start >= total_found: break

        shard_file = os.path.join(SHARDS_DIR, f"shard_{shard_idx + 1}.parquet")
        shard_paths.append(shard_file)
        
        if os.path.exists(shard_file) and not FORCE_REBUILD:
            print(f"Exists: {shard_file}")
            continue

        print(f"Building {shard_file} (Stream mode)...")
        writer = pq.ParquetWriter(shard_file, schema)
        failed_ranges = []
        total_shard_rows = 0

        # Process function
        def process_range(start, retmax):
            # 1. Fetch
            seqs = fetch_batch_sequences(start, retmax, webenv, query_key)
            if not seqs: return False
            meta = fetch_batch_metadata(start, retmax, webenv, query_key)
            if not meta: return False

            # 2. Map
            acc_map = {}
            for d in meta:
                tid = str(d.get("TaxId", "Unknown"))
                if d.get("AccessionVersion"): acc_map[d["AccessionVersion"].split('.')[0]] = tid
                if d.get("Caption"): acc_map[d["Caption"].split('.')[0]] = tid
            
            # 3. Taxonomy
            tids = list({t for t in acc_map.values() if t != "Unknown"})
            tax_lookup = fetch_taxonomy_metadata(tids)
            
            # 4. Build Rows
            rows = []
            for s in seqs:
                seq_str = str(s.seq).upper()
                if not (200 <= len(seq_str) <= 2000): continue
                
                acc = s.id.split('.')[0]
                tx_id = acc_map.get(acc, "Unknown")
                tx_info = tax_lookup.get(tx_id, _rank_aware_fill({}))
                desc = s.description.split(' ', 1)
                
                rows.append({
                    "AccessionID": acc,
                    "ScientificName": desc[1] if len(desc)>1 else "Unknown",
                    "TaxID": tx_id,
                    **tx_info,
                    "Sequence": seq_str,
                    "metagenomic_source": "NCBI-Nucleotide",
                    "Quality_Check": True
                })
            
            # Cleanup source data immediately
            del seqs, meta, acc_map, tids, tax_lookup
            gc.collect()

            # 5. Write
            if rows:
                rb = pa.RecordBatch.from_pandas(pd.DataFrame(rows), schema=schema)
                writer.write_batch(rb)
                c = len(rows)
                del rows, rb
                return c
            return 0

        # Main Loop
        for start in range(shard_start, shard_end, FETCH_BATCH_SIZE):
            rmax = min(FETCH_BATCH_SIZE, shard_end - start)
            count = process_range(start, rmax)
            
            if count is False:
                failed_ranges.append((start, rmax))
            else:
                total_shard_rows += count
                if total_shard_rows % 1000 == 0:
                    print(f"  Written {total_shard_rows} rows...", end="\r")
            
            time.sleep(1.0) # Slower pace to prevent overlap/buffer issues

        # Retry Loop
        if failed_ranges:
            print(f"\nRetrying {len(failed_ranges)} ranges...")
            for s, r in failed_ranges:
                half = r // 2
                c1 = process_range(s, half)
                if c1: total_shard_rows += c1
                time.sleep(1.0)
                
                c2 = process_range(s + half, r - half)
                if c2: total_shard_rows += c2
                time.sleep(1.0)
        
        writer.close()
        print(f"\nSaved {shard_file}: {total_shard_rows} records.")
        gc.collect()
        
    return shard_paths

shard_files = fetch_marine_eukaryotes_500k()


In [ ]:
# @Embedder-ML: Step 2 - Neural Embedding Pipeline (500k Ready)
import torch.nn.functional as F

class LargeScaleEmbedder:
    def __init__(self, model_name="InstaDeepAI/nucleotide-transformer-v2-50m-multi-species"):
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        gpu_name = torch.cuda.get_device_name(0) if self.device == "cuda" else "CPU"
        self.batch_size = 256 if "A100" in gpu_name.upper() else 128
        print(f"Initializing {model_name} on {self.device} ({gpu_name})")
        print(f"Auto batch size: {self.batch_size}")

        config = AutoConfig.from_pretrained(model_name, trust_remote_code=True)
        config.intermediate_size = 4096
        
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForMaskedLM.from_pretrained(
            model_name,
            config=config,
            trust_remote_code=True,
            ignore_mismatched_sizes=True
        ).to(self.device)
        self.model.eval()

    @staticmethod
    def l2_normalize(vectors: np.ndarray, eps: float = 1e-12) -> np.ndarray:
        norms = np.linalg.norm(vectors, axis=1, keepdims=True)
        norms = np.clip(norms, eps, None)
        return (vectors / norms).astype(np.float32)

    def embedding_generator(self, sequences, batch_size=None):
        """Yields L2-normalized 768-d float32 vectors."""
        bs = batch_size or self.batch_size

        for i in tqdm(range(0, len(sequences), bs), desc="Embedding Batches"):
            batch = sequences[i:i + bs]
            batch = [seq.upper().replace("\n", "").replace("\r", "").replace("N", "A")[:1000] for seq in batch]

            inputs = self.tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=1000
            ).to(self.device)

            with torch.no_grad():
                outputs = self.model(**inputs, output_hidden_states=True)
                last_hidden_state = outputs.hidden_states[-1]
                attention_mask = inputs["attention_mask"].unsqueeze(-1).expand(last_hidden_state.size()).float()

                sum_embeddings = torch.sum(last_hidden_state * attention_mask, 1)
                sum_mask = torch.clamp(attention_mask.sum(1), min=1e-9)
                mean_embeddings = sum_embeddings / sum_mask

                current_dim = mean_embeddings.shape[1]
                target_dim = 768
                if current_dim < target_dim:
                    padding = torch.zeros((mean_embeddings.shape[0], target_dim - current_dim), device=self.device)
                    mean_embeddings = torch.cat([mean_embeddings, padding], dim=1)
                elif current_dim > target_dim:
                    mean_embeddings = mean_embeddings[:, :target_dim]

                vectors = mean_embeddings.detach().cpu().numpy().astype(np.float32)
                vectors = self.l2_normalize(vectors)

            del inputs, outputs, last_hidden_state, attention_mask, sum_embeddings, sum_mask, mean_embeddings
            if self.device == "cuda":
                torch.cuda.empty_cache()

            yield vectors

embedder = LargeScaleEmbedder()

In [ ]:
# @Data-Ops: Step 3 - Embed, enforce schema, and export mandatory atlas parquet
# TITANIUM STABILITY MODE: Single-Thread, Tiny Batches, Aggressive GC
import gc
import sys
import pyarrow as pa
import pyarrow.parquet as pq
import numpy as np
import torch
import os

# Safety check for kernel restart
if 'embedder' not in globals():
    raise RuntimeError("⚠️ Kernel was restarted! You MUST re-run 'Step 2 - Neural Embedding Pipeline' cell before running this cell.")

print("Embedding with TITANIUM STABILITY settings to reference_atlas_500k.parquet...")

required_cols = [
    "AccessionID", "ScientificName", "TaxID",
    "Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species",
    "Sequence", "metagenomic_source", "Quality_Check", "vector"
]

final_output = "reference_atlas_500k.parquet"
shard_files = [os.path.join("shards", f"shard_{i}.parquet") for i in range(1, 6)]

if os.path.exists(final_output):
    os.remove(final_output)

writer = None
total_records = 0

# --- EXTREME STABILITY CONFIG ---
# Read only 100 rows from disk at a time (Minimal RAM footprint)
MICRO_BATCH_SIZE = 100  
# Process 4 sequences on GPU at a time (Minimal VRAM footprint)
GPU_BATCH_SIZE = 4      

for shard_file in shard_files:
    if not os.path.exists(shard_file):
        print(f"Skipping missing shard: {shard_file}")
        continue

    print(f"Streaming {shard_file}...")
    
    try:
        parquet_file = pq.ParquetFile(shard_file)
        
        # iter_batches with use_threads=False is critical for low RAM
        for batch in parquet_file.iter_batches(batch_size=MICRO_BATCH_SIZE, use_threads=False):
            # Explicit garbage collection before new allocation
            gc.collect()
            
            df_chunk = batch.to_pandas()
            
            # 1. Schema guard
            for col in required_cols:
                if col == "vector": continue 
                if col not in df_chunk.columns:
                    if col in {"Kingdom", "Phylum", "Class", "Order", "Family", "Genus", "Species"}:
                        df_chunk[col] = f"[Unclassified {col}]"
                    elif col == "metagenomic_source":
                        df_chunk[col] = "NCBI-Nucleotide"
                    else:
                        df_chunk[col] = "Unknown"

            # 2. Extract sequences
            seqs = df_chunk["Sequence"].tolist()
            
            # 3. RUN EMBEDDING
            vectors_list = []
            try:
                # Embedding loop with explicit cleanup
                for batch_vectors in embedder.embedding_generator(seqs, batch_size=GPU_BATCH_SIZE):
                    vectors_list.append(batch_vectors)
                    # Clean CUDA cache frequently
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
            except Exception as e:
                print(f"⚠️ GPU ERROR in batch: {e}")
                del seqs
                del df_chunk
                gc.collect()
                continue # Skip this micro-batch to save the session

            del seqs
            
            if not vectors_list:
                del df_chunk
                continue

            # Concatenate
            try:
                final_vectors = np.concatenate(vectors_list, axis=0).astype(np.float32)
            except ValueError:
                del vectors_list
                del df_chunk
                continue
                
            del vectors_list
            
            # Normalize
            final_vectors = embedder.l2_normalize(final_vectors)
            
            # Assign
            df_chunk["vector"] = list(final_vectors)
            df_chunk["metagenomic_source"] = "NCBI-Nucleotide"
            del final_vectors

            # 4. Write
            table_chunk = pa.Table.from_pandas(df_chunk, preserve_index=False)
            del df_chunk
            
            if writer is None:
                writer = pq.ParquetWriter(final_output, table_chunk.schema)
            
            writer.write_table(table_chunk)
            total_records += len(table_chunk)
            del table_chunk
            
            # 5. Flush
            print(f"  Processed {total_records:,} records...", end="\r")
            
    except Exception as e:
        print(f"\nCRITICAL ERROR processing {shard_file}: {e}")
        # Dont raise, try next shard to save progress? 
        # Actually safer to continue and save what we have
        continue

    print(f"\nShard {shard_file} complete.")
    gc.collect()

if writer:
    writer.close()
    print(f"\nSUCCESS: {final_output} created with {total_records:,} records.")
else:
    print("\nNo data processed.")

print("Schema validation complete.")
